# **Обучение**

In [1]:
!find /kaggle/input /kaggle/working -type f \( -name train_script.py -o -name baseline_plus_plus_longer.yaml -o -name dataset_dict.json -o -name best.pt -o -name "*.zip" \)

/kaggle/input/datasets/wwip62/nlp-hw2-runs/baseline_plus_plus_kaggle/checkpoints/best.pt


/kaggle/input/datasets/wwip62/nlp-hw2-codegen/experiment_configs/baseline_plus_plus_longer.yaml


/kaggle/input/datasets/wwip62/nlp-hw2-codegen/utilities/train_script.py


/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized/dataset_dict.json


In [2]:
from pathlib import Path

CODE_DIR = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen")
DATA_SOURCE = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized")
WEIGHTS_FROM = Path("/kaggle/input/datasets/wwip62/nlp-hw2-runs/baseline_plus_plus_kaggle/checkpoints/best.pt")

CONFIG_SOURCE = CODE_DIR / "experiment_configs/baseline_plus_plus_longer.yaml"
WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "tokenized"
CONFIG_PATH = WORK_DIR / "train_config.yaml"

RUN_NAME = "baseline_plus_plus_longer"
NUM_GPUS = 2

LR = 0.00015
MIN_LR = 0.000015
WARMUP_STEPS = 5
WARMUP_START_FACTOR = 0.2

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"

In [3]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

import os
import shutil
import yaml
from kaggle_secrets import UserSecretsClient

DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_SOURCE / "dataset_dict.json", DATA_DIR / "dataset_dict.json")
shutil.copytree(DATA_SOURCE / "val", DATA_DIR / "val", dirs_exist_ok=True)

if not (DATA_DIR / "train").exists():
    (DATA_DIR / "train").symlink_to(DATA_SOURCE / "train", target_is_directory=True)

config = yaml.safe_load(CONFIG_SOURCE.read_text())
config["name"] = RUN_NAME
config["data"]["tokenized_dir"] = str(DATA_DIR)
config["data"]["tokenizer_path"] = str(DATA_SOURCE / "tokenizer.json")
config["optimizer"]["params"]["lr"] = LR
config["scheduler"]["params"]["eta_min"] = MIN_LR
config["scheduler"]["warmup"] = {
    "steps": WARMUP_STEPS,
    "start_factor": WARMUP_START_FACTOR,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))

env = os.environ.copy()
env["PROJECT_DIR"] = str(WORK_DIR)
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

print("Подготовка завершена.")
print("Результаты:", WORK_DIR / "experiment_runs" / RUN_NAME)

Note: you may need to restart the kernel to use updated packages.


Подготовка завершена.
Результаты: /kaggle/working/experiment_runs/baseline_plus_plus_longer


In [4]:
import subprocess
import sys
import torch

assert torch.cuda.device_count() >= NUM_GPUS, "Включи GPU T4 ×2 в настройках Kaggle."

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={NUM_GPUS}",
    "-m", "utilities.train_script",
    "--config", str(CONFIG_PATH),
    "--weights-from", str(WEIGHTS_FROM),
    "--wandb-project", WANDB_PROJECT,
]

if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]

subprocess.run(command, cwd=CODE_DIR, env=env, check=True)


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************


Flattening the indices: 100%|██████████| 128/128 [00:00<00:00, 11019.74 examples/s]


Run: /kaggle/working/experiment_runs/baseline_plus_plus_longer; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

Epoch 2, mini-epoch 31 | train loss = 0.2271, val loss = 0.2170
Examples: 14,610,496 | time = 190.6 s (train 163.3, val 27.3) | train/example = 0.001634 s
Small validation: token accuracy = 93.30%, top-5 = 99.24%
Heavy validation: pass@1 = 65.62% (84 / 128) | time = 19.0 s
Syntax valid = 100.00%, numeric return = 98.44%, no EOS = 1.56%
BLEU = 51.35, repetition = 6.03%, timeout = 0.00%


Epoch 2, mini-epoch 32 | train loss = 0.2274, val loss = 0.2164
Examples: 14,710,562 | time = 193.2 s (train 165.7, val 27.5) | train/example = 0.001656 s
Small validation: token accuracy = 93.28%, top-5 = 99.24%
Heavy validation: pass@1 = 69.53% (89 / 128) | time = 19.2 s
Syntax valid = 100.00%, numeric return = 99.22%, no EOS = 0.78%
BLEU = 52.66, repetition = 5.39%, timeout = 0.00%


Epoch 2, mini-epoch 33 | train loss = 0.2268, val loss = 0.2165
Examples: 14,810,580 | time = 192.4 s (train 164.8, val 27.6) | train/example = 0.001648 s
Small validation: token accuracy = 93.29%, top-5 = 99.24%
Heavy validation: pass@1 = 69.53% (89 / 128) | time = 19.3 s
Syntax valid = 100.00%, numeric return = 99.22%, no EOS = 0.78%
BLEU = 51.34, repetition = 5.54%, timeout = 0.00%


Epoch 2, mini-epoch 34 | train loss = 0.2269, val loss = 0.2160
Examples: 14,910,575 | time = 193.1 s (train 166.0, val 27.1) | train/example = 0.001660 s
Small validation: token accuracy = 93.30%, top-5 = 99.26%
Heavy validation: pass@1 = 70.31% (90 / 128) | time = 18.9 s
Syntax valid = 100.00%, numeric return = 99.22%, no EOS = 0.78%
BLEU = 49.38, repetition = 5.23%, timeout = 0.00%


Epoch 2, mini-epoch 35 | train loss = 0.2261, val loss = 0.2163
Examples: 15,010,551 | time = 191.2 s (train 164.1, val 27.1) | train/example = 0.001641 s
Small validation: token accuracy = 93.31%, top-5 = 99.24%
Heavy validation: pass@1 = 70.31% (90 / 128) | time = 18.8 s
Syntax valid = 100.00%, numeric return = 98.44%, no EOS = 1.56%
BLEU = 49.95, repetition = 5.24%, timeout = 0.00%
